# GRPO Training with Oracle Reward Model (Async)

This notebook trains a policy model using **Group Relative Policy Optimization (GRPO)** with an external oracle (GPT-4o-mini) as the reward model.
The oracle uses **structured JSON-schema outputs** and runs **asynchronously** for throughput.

---

## Experiment-Based Architecture

This notebook supports two modes controlled by `EXPLORATION_MODE`:

### 🚀 Training Mode (`EXPLORATION_MODE = False`)
- **Auto-resume**: Automatically resumes from the latest valid checkpoint if available
- **Fresh start**: Starts training from scratch if no checkpoints exist
- **Versioning**: Every epoch is saved as a checkpoint (no limit)
- **Metadata**: Writes `experiment_metadata.json` alongside each checkpoint
- **W&B resume**: Consistent run IDs enable resume across restarts

### 🔍 Exploration Mode (`EXPLORATION_MODE = True`)
- **Safe**: Training loop is **completely skipped**
- **Read-only**: Uses saved checkpoints for evaluation only
- **Multi-adapter sweep**: Loads many adapters once and switches with `set_adapter()` / `disable_adapter()`
- **Flexible**: Sweep any 0-indexed epoch list by setting indices

---

## Notebook Flow

1. **Configuration** — Experiment name, mode selection, hyperparameters, quick-test overrides  
2. **Setup** — Installs, imports, authentication, async patching, seeds  
3. **Tokenizer** — Load tokenizer and configure ChatML-style chat template  
4. **Model Helpers** — Checkpoint detection & integrity checks, model loading, generation patching  
5. **Data** — Load CSVs, filter, format prompts, create transcripts for oracle, enforce stop strings  
6. **Quantization & LoRA** — 4-bit quantization and LoRA adapter configuration  
7. **Oracle Reward** — Async GPT-based reward with JSON-schema validation  
8. **Policy** — Load base model + adapter (mode-aware)  
9. **Training** — GRPO trainer with W&B logging + checkpoint metadata (skipped in Exploration Mode)  
10. **Exploration Helpers** — Multi-adapter sweep utilities  
11. **Post-Training Evaluation** — Test prompt sampling + sweep run

---
## 1. Configuration

All tunable parameters in one place.
- Toggle `QUICK_TEST_MODE` for minimal settings.
- Toggle `EXPLORATION_MODE` to switch between training and read-only evaluation.
- Derived names (e.g., `EXPERIMENT_NAME`, `CURRENT_ADAPTER_REPO`) are computed below.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                              CONFIGURATION                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────────
# Quick Test & Exploration Mode (set True for fast iteration or eval-only sweep)
# ─────────────────────────────────────────────────────────────────────────────────
QUICK_TEST_MODE = False  # Toggle for fast smoke tests
EXPLORATION_MODE = True  # True = exploration (no training), False = training

# ─────────────────────────────────────────────────────────────────────────────────
# Model IDs
# ─────────────────────────────────────────────────────────────────────────────────
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B"
TOKENIZER_ID = "meta-llama/Llama-3.2-1B"
ORACLE_MODEL_ID = "gpt-4o-mini-2024-07-18"

# ─────────────────────────────────────────────────────────────────────────────────
# Experiment Configuration
# ─────────────────────────────────────────────────────────────────────────────────
EXPERIMENT_NAME_BASE = "GRPO_Oracle_Llama32-1B-Instruct_LA5_G4_V2"  # Base experiment name
HUB_ENTITY = "LBK95"  # Your HuggingFace username

# ─────────────────────────────────────────────────────────────────────────────────
# Data Configuration
# ─────────────────────────────────────────────────────────────────────────────────
DATA_SUBPATH = "LLM_DATA/Conversation_Trees_GRPO_V2/LookAhead_5/TTree1.2_TT0.9_TP0.7_TE0.2_V1"
NUM_DATA_FILES = 96 if not EXPLORATION_MODE else 5  # Number of pref_data_{i}.csv files to load
SCORE_THRESHOLD = 0.1  # Minimum score difference for preference pairs
MIN_MESSAGES_LENGTH = 0  # Minimum number of messages in conversation (filters out short conversations)
EVAL_SPLIT_RATIO = 0.05  # Fraction of data for evaluation

# ─────────────────────────────────────────────────────────────────────────────────
# Prompt & Generation
# ─────────────────────────────────────────────────────────────────────────────────
STOP_STRINGS = ["<|im_end|>", "<|im_start|>"]  # Add more stop strings as needed
STOP_STRING_PRIMARY = STOP_STRINGS[0]
MAX_ALLOWED_PROMPT_LENGTH = 4096
MAX_COMPLETION_LENGTH = 200
DEFAULT_TEMPERATURE = 0.9

# ─────────────────────────────────────────────────────────────────────────────────
# Oracle Reward
# ─────────────────────────────────────────────────────────────────────────────────
QUESTIONNAIRE_IDS = [1, 2]  # Questionnaires used for reward evaluation
ORACLE_MAX_CONCURRENCY = 8  # Max parallel API calls
ORACLE_MAX_RETRIES = 3
EVAL_TEMPERATURE = 0.0

# ─────────────────────────────────────────────────────────────────────────────────
# Training Hyperparameters
# ─────────────────────────────────────────────────────────────────────────────────
LEARNING_RATE = 1e-5  # Lower default for stability during restarts
NUM_TRAIN_EPOCHS = 24 # 15 -> 24
RESTART_EVERY_N_EPOCHS = 3  # Cosine LR restarts every N epochs
NUM_CYCLES = NUM_TRAIN_EPOCHS // RESTART_EVERY_N_EPOCHS  # Number of cosine cycles
WARMUP_RATIO = 0.01  # Warmup fraction of total training steps
NUM_GENERATIONS = 4  # Completions per prompt for GRPO
GRPO_BETA = 0.01
TRAIN_BATCH_SIZE = 8 # must be divisible by NUM_GENERATIONS
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1
GRPO_TEMPERATURE = 1.2
assert TRAIN_BATCH_SIZE % NUM_GENERATIONS == 0, "TRAIN_BATCH_SIZE must be divisible by NUM_GENERATIONS"
assert EVAL_BATCH_SIZE % NUM_GENERATIONS == 0, "EVAL_BATCH_SIZE must be divisible by NUM_GENERATIONS"
assert NUM_TRAIN_EPOCHS % RESTART_EVERY_N_EPOCHS == 0, "NUM_TRAIN_EPOCHS must be divisible by RESTART_EVERY_N_EPOCHS"

# ─────────────────────────────────────────────────────────────────────────────────
# LoRA Configuration
# ─────────────────────────────────────────────────────────────────────────────────
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"]

# ─────────────────────────────────────────────────────────────────────────────────
# Logging & Checkpointing
# ─────────────────────────────────────────────────────────────────────────────────
LOGGING_STEPS = 1
SAVE_STRATEGY = "epoch"
SAVE_TOTAL_LIMIT = None  # Keep all checkpoints (epochs)
PUSH_TO_HUB = True
LOG_COMPLETIONS = True
REPORT_TO = "wandb"

# ─────────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────────────────────────
SEED = 42

# ═══════════════════════════════════════════════════════════════════════════════
# Quick Test Mode Overrides
# ═══════════════════════════════════════════════════════════════════════════════
if QUICK_TEST_MODE:
    NUM_DATA_FILES = 5           
    NUM_TRAIN_EPOCHS = 1         
    NUM_GENERATIONS = 4          
    MAX_COMPLETION_LENGTH = 64   
    TRAIN_BATCH_SIZE = 2
    EVAL_BATCH_SIZE = 2
    PUSH_TO_HUB = False
    LOG_COMPLETIONS = False
    REPORT_TO = "none"
    print("⚡ QUICK_TEST_MODE enabled — using minimal settings")

# ═══════════════════════════════════════════════════════════════════════════════
# Derived Configuration (do not edit)
# ═══════════════════════════════════════════════════════════════════════════════

# Determine mode
TRAINING_MODE = not EXPLORATION_MODE

# Apply suffix for quick test mode — affects ALL names consistently
RUN_SUFFIX = "_QUICKTEST" if QUICK_TEST_MODE else ""
EXPERIMENT_NAME = f"{EXPERIMENT_NAME_BASE}{RUN_SUFFIX}"  # Effective experiment name

# Construct paths using effective EXPERIMENT_NAME (already includes suffix)
CURRENT_ADAPTER_REPO = f"{HUB_ENTITY}/{EXPERIMENT_NAME}"

print("=" * 70)
print("CONFIGURATION SUMMARY")
print("=" * 70)
print(f"  Experiment:         {EXPERIMENT_NAME}")
print(f"  Base model:         {BASE_MODEL_ID}")
print(f"  Oracle model:       {ORACLE_MODEL_ID}")
print(f"  Adapter repo:       {CURRENT_ADAPTER_REPO}")
print(f"  Quick test mode:    {QUICK_TEST_MODE}")
print("-" * 70)
if EXPLORATION_MODE:
    print("  🔍 MODE: EXPLORATION")
    print("     Training will be SKIPPED. Proceeding to evaluation only.")
else:

    print("  🚀 MODE: TRAINING (auto-resume enabled)")
    print("=" * 70)
    print(f"     Epochs: {NUM_TRAIN_EPOCHS} | Batch: {TRAIN_BATCH_SIZE} | Push to Hub: {PUSH_TO_HUB}")

CONFIGURATION SUMMARY
  Experiment:         GRPO_Oracle_Llama32-1B-Instruct_LA5_G4_V2
  Base model:         meta-llama/Llama-3.2-1B
  Oracle model:       gpt-4o-mini-2024-07-18
  Adapter repo:       LBK95/GRPO_Oracle_Llama32-1B-Instruct_LA5_G4_V2
  Quick test mode:    False
----------------------------------------------------------------------
  🔍 MODE: EXPLORATION
     Training will be SKIPPED. Proceeding to evaluation only.


---
## 2. Setup

Installs (optional), imports, async event loop patching, authentication, and reproducibility seeds.
Also prints versions for key libraries to aid debugging.

In [2]:
# Uncomment to install dependencies (run once)
# %pip install -U -q transformers accelerate datasets peft bitsandbytes trl
# %pip install weave wandb hf_xet nest_asyncio

import nest_asyncio
nest_asyncio.apply()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────────
# Standard Library
# ─────────────────────────────────────────────────────────────────────────────────
import os
import sys
import re
import ast
import json
import types
import random
import asyncio
import tempfile

# ─────────────────────────────────────────────────────────────────────────────────
# Third-Party Libraries
# ─────────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import wandb
import weave
import datasets
from datasets import Dataset, load_dataset
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
import peft
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
import trl 
from trl import GRPOTrainer, GRPOConfig
from openai import AsyncOpenAI
from huggingface_hub import HfApi, repo_exists

# ─────────────────────────────────────────────────────────────────────────────────
# Resolve shared modules from workspace root
# ─────────────────────────────────────────────────────────────────────────────────
# Walk up from CWD to find the workspace root (depth-independent).
_HELPER_FILES = ['openai_key.txt', 'HF_key.txt']
_cur = os.path.abspath(os.getcwd())
_WORKSPACE_ROOT = None
for _ in range(8):
    if all(os.path.exists(os.path.join(_cur, _hf)) for _hf in _HELPER_FILES):
        _WORKSPACE_ROOT = _cur
        break
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        break
    _cur = _parent
if _WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate workspace root with key files')
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)

# ─────────────────────────────────────────────────────────────────────────────────
# Local Imports
# ─────────────────────────────────────────────────────────────────────────────────
from questionnaires_V5 import get_prompt_eval_questionnaire

# ─────────────────────────────────────────────────────────────────────────────────
# Environment Detection
# ─────────────────────────────────────────────────────────────────────────────────
IN_COLAB = 'COLAB_GPU' in os.environ
print(f"Running in Colab: {IN_COLAB}")

# ─────────────────────────────────────────────────────────────────────────────────
# Set Seeds for Reproducibility
# ─────────────────────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"✓ Seeds set (SEED={SEED})")


# ─────────────────────────────────────────────────────────────────────────────────
# Print Version Info for Key Libraries
# ─────────────────────────────────────────────────────────────────────────────────
print(f"transformers version: {transformers.__version__}")
print(f"datasets version:     {datasets.__version__}")
print(f"peft version:         {peft.__version__}")
print(f"trl version:          {trl.__version__}")
print("✓ Version info printed")

ModuleNotFoundError: No module named 'wandb'

### 2.1 Authentication (OpenAI, HuggingFace, W&B)

In [4]:
import os
import sys
# Walk up from CWD to find the workspace root (depth-independent).
_HELPER_FILES = ['openai_key.txt', 'HF_key.txt']
_cur = os.path.abspath(os.getcwd())
_WORKSPACE_ROOT = None
for _ in range(8):
    if all(os.path.exists(os.path.join(_cur, _hf)) for _hf in _HELPER_FILES):
        _WORKSPACE_ROOT = _cur
        break
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        break
    _cur = _parent
if _WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate workspace root with key files')
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)
# Load OpenAI API key
if IN_COLAB:
    from google.colab import userdata
    OpenAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()
else:
    OpenAI_API_KEY = open(os.path.join(_WORKSPACE_ROOT, "openai_key.txt"), "r").read().strip()

client = AsyncOpenAI(api_key=OpenAI_API_KEY)
print("✓ OpenAI client initialized")

✓ OpenAI client initialized


In [5]:
# Colab-specific: HuggingFace, W&B login, and Google Drive mount
if IN_COLAB:
    from huggingface_hub import login
    from google.colab import userdata, drive
    
    login(token=userdata.get("huggingface"))
    print("✓ Logged in to Hugging Face Hub")
    
    wandb.login(key=userdata.get("wandb"))
    print("✓ Logged in to Weights & Biases")
    
    drive.mount('/content/drive')
    print("✓ Mounted Google Drive")
else:
    print("(Skipped Colab-specific auth — running locally)")

(Skipped Colab-specific auth — running locally)


---
## 3. Tokenizer

Load tokenizer and configure a custom **ChatML-style chat template** with `<|im_start|>` / `<|im_end|>` delimiters.

In [6]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.truncation_side = "left"

# Custom ChatML-style template
tokenizer.chat_template = (
    "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}"
    "{% for message in messages %}"
    "{{'<|im_start|> ' + message['role'] + '\n' + message['content'] + ' <|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|> assistant\n' }}{% endif %}"
)

print(f"✓ Tokenizer loaded: {TOKENIZER_ID}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")

✓ Tokenizer loaded: meta-llama/Llama-3.2-1B
  Vocab size: 128256
  Pad token: <|end_of_text|> (id=128001)


In [7]:
# Test chat template (optional verification)
_test_msgs = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
]
_test_prompt = tokenizer.apply_chat_template(_test_msgs, add_generation_prompt=True, tokenize=False)
print("Sample formatted prompt:")
print(_test_prompt)

Sample formatted prompt:
<|im_start|> system
You are a helpful assistant. <|im_end|>
<|im_start|> user
Hello! <|im_end|>
<|im_start|> assistant



---
## 4. Model Helpers

Utilities for checkpoint discovery + integrity checks, base model loading, parameter counting, and generation patching (to enforce stop strings).

In [8]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                           MODEL HELPER FUNCTIONS                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────────
# Token Sync
# ─────────────────────────────────────────────────────────────────────────────────

def sync_pad_token(model, tok):
    """Synchronize pad/eos/bos token IDs between model config and tokenizer."""
    model.config.pad_token_id = tok.pad_token_id
    model.config.eos_token_id = tok.eos_token_id
    model.config.bos_token_id = tok.bos_token_id
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.pad_token_id = tok.pad_token_id
        model.generation_config.eos_token_id = tok.eos_token_id
        model.generation_config.bos_token_id = tok.bos_token_id


# ─────────────────────────────────────────────────────────────────────────────────
# Checkpoint Detection
# ─────────────────────────────────────────────────────────────────────────────────

def list_local_checkpoints(output_dir: str) -> list[str]:
    """List all checkpoint directories sorted by step."""
    if not os.path.isdir(output_dir):
        return []
    checkpoints = []
    for name in os.listdir(output_dir):
        if name.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, name)):
            try:
                step = int(name.split("-")[-1])
                checkpoints.append((step, os.path.join(output_dir, name)))
            except ValueError:
                continue
    checkpoints.sort(key=lambda x: x[0])
    return [path for _, path in checkpoints]


def get_latest_checkpoint(output_dir: str) -> str | None:
    """Find the latest checkpoint directory by step number."""
    checkpoints = list_local_checkpoints(output_dir)
    return checkpoints[-1] if checkpoints else None


def get_checkpoint_by_identifier(output_dir: str, identifier) -> str | None:
    """
    Find a specific checkpoint by identifier.
    
    Args:
        output_dir: Directory containing checkpoints
        identifier: Can be:
            - str like "checkpoint-500" (exact name)
            - int like 2 (epoch index, 1-indexed — returns 2nd checkpoint)
    
    Returns:
        Full path to the checkpoint directory, or None if not found.
    """
    if not os.path.isdir(output_dir):
        return None
    
    # String: try exact match
    if isinstance(identifier, str):
        exact_path = os.path.join(output_dir, identifier)
        if os.path.isdir(exact_path):
            return exact_path
        # Try adding "checkpoint-" prefix
        if not identifier.startswith("checkpoint-"):
            prefixed_path = os.path.join(output_dir, f"checkpoint-{identifier}")
            if os.path.isdir(prefixed_path):
                return prefixed_path
        return None
    
    # Integer: treat as 1-indexed epoch position
    if isinstance(identifier, int):
        checkpoints = list_local_checkpoints(output_dir)
        if 1 <= identifier <= len(checkpoints):
            return checkpoints[identifier - 1]
        return None
    
    return None


def validate_checkpoint_integrity(checkpoint_path: str) -> tuple[bool, list[str]]:
    """
    Validate that a checkpoint has required files for resuming.
    
    Returns:
        Tuple of (is_valid, list_of_missing_files)
    """
    required_files = [
        "adapter_model.safetensors",
        "trainer_state.json",
    ]
    missing = [f for f in required_files if not os.path.exists(os.path.join(checkpoint_path, f))]
    return len(missing) == 0, missing


# ─────────────────────────────────────────────────────────────────────────────────
# Parameter Counting
# ─────────────────────────────────────────────────────────────────────────────────

def get_adapter_param_count(model) -> dict:
    """Get trainable vs total parameter counts for a model."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {
        "trainable": trainable,
        "total": total,
        "trainable_pct": 100 * trainable / total if total > 0 else 0,
        "trainable_M": trainable / 1e6,
        "total_M": total / 1e6,
    }


# ─────────────────────────────────────────────────────────────────────────────────
# Model Loading
# ─────────────────────────────────────────────────────────────────────────────────

def load_base_policy(quantization_config=None, use_fp16_fallback=False):
    """
    Load the base policy model with 4-bit quantization (or fp16 fallback).
    Returns the model prepared for training (NO adapter attached yet).
    """
    if use_fp16_fallback:
        print("  Loading base model in fp16/bf16 (fallback mode)...")
        compute_dtype = (
            torch.bfloat16 
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported() 
            else torch.float16
        )
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map="auto",
            dtype=compute_dtype,
        )
    else:
        print("  Loading base model with 4-bit quantization...")
        if quantization_config is None:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=(
                    torch.bfloat16 
                    if torch.cuda.is_available() and torch.cuda.is_bf16_supported() 
                    else torch.float16
)
,
            )
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map="auto",
            quantization_config=quantization_config,
            dtype=quantization_config.bnb_4bit_compute_dtype,
        )
    
    model.config.use_cache = False
    if not use_fp16_fallback:
        model = prepare_model_for_kbit_training(model)
    sync_pad_token(model, tokenizer)
    
    model.generation_config.temperature = DEFAULT_TEMPERATURE
    model.generation_config.stop_strings = STOP_STRINGS
    
    return model


def load_policy_from_checkpoint(base_policy, output_dir, checkpoint_id):
    """
    Load base or adapter checkpoint for evaluation/training reuse.
    checkpoint_id: 0/"base"/"checkpoint-0" for base, or checkpoint name/int.
    
    NOTE: This function creates a NEW PeftModel each time for non-base checkpoints.
    For multi-checkpoint sweeps, prefer build_multi_adapter_model() to avoid
    mutating base_policy repeatedly.
    """
    if checkpoint_id in (0, "0", "base", "BASE", "checkpoint-0"):
        print("Using base model (no adapter).")
        model = base_policy
    else:
        ckpt_path = get_checkpoint_by_identifier(output_dir, checkpoint_id)
        if ckpt_path is None:
            available = [os.path.basename(c) for c in list_local_checkpoints(output_dir)]
            raise FileNotFoundError(
                f"Checkpoint '{checkpoint_id}' not found! "
                f"Available: {available}"
            )
        is_valid, missing = validate_checkpoint_integrity(ckpt_path)
        if not is_valid:
            raise RuntimeError(f"Checkpoint incomplete! Missing: {missing}")
        print(f"Loading checkpoint: {os.path.basename(ckpt_path)}")
        model = PeftModel.from_pretrained(base_policy, ckpt_path, is_trainable=False)

    patch_generate(model)
    return model


def patch_generate(model):
    """Patch model.generate() to auto-inject tokenizer (required for stop_strings)."""
    if hasattr(model, "_generate_patched"):
        return
    
    if not hasattr(model, "_original_generate"):
        model._original_generate = model.generate
    
    def generate_with_tokenizer(self, *args, **kwargs):
        if "tokenizer" not in kwargs:
            kwargs["tokenizer"] = tokenizer
        return self._original_generate(*args, **kwargs)
    
    model.generate = types.MethodType(generate_with_tokenizer, model)
    model._generate_patched = True
    print(f"  ✓ Patched generate() for {type(model).__name__}")


print("✓ Model helper functions defined")

✓ Model helper functions defined


---
## 5. Data Loading & Preparation

- Load preference data CSVs (`pref_data_{i}.csv`)
- Filter by minimum conversation length and score threshold
- Apply chat template to create **prompts** (for model input)
- Create **transcripts** (plain text with `[PATIENT]`/`[THERAPIST]` labels for oracle evaluation)
- Append stop strings to responses to enforce generation boundaries
- Handle prompt length truncation
- Create HuggingFace Dataset with train/eval split

In [9]:
# ─────────────────────────────────────────────────────────────────────────────────
# Resolve Data Path
# ─────────────────────────────────────────────────────────────────────────────────
PREF_DATA_PATH = f"/content/drive/MyDrive/{DATA_SUBPATH}" if IN_COLAB else DATA_SUBPATH

print(f"Data path:       {PREF_DATA_PATH}")
print(f"Files to load:   {NUM_DATA_FILES}")
print(f"Score threshold: {SCORE_THRESHOLD}")

Data path:       LLM_DATA/Conversation_Trees_GRPO_V2/LookAhead_5/TTree1.2_TT0.9_TP0.7_TE0.2_V1
Files to load:   5
Score threshold: 0.1


In [10]:
# ─────────────────────────────────────────────────────────────────────────────────
# Load CSV Files
# ─────────────────────────────────────────────────────────────────────────────────
pref_datasets = []
missing_files = []

for i in range(NUM_DATA_FILES):
    file_path = f"{PREF_DATA_PATH}/pref_data_{i}.csv"
    if os.path.exists(file_path):
        pref_datasets.append(pd.read_csv(file_path))
    else:
        missing_files.append(i)

# Validation: ensure we have data
assert len(pref_datasets) > 0, f"No data files found in {PREF_DATA_PATH}"

full_pref_data = pd.concat(pref_datasets, ignore_index=True)
full_pref_data["messages"] = full_pref_data["messages"].apply(ast.literal_eval)


print(f"✓ Loaded {len(pref_datasets)}/{NUM_DATA_FILES} files")
print(f"  Total samples: {len(full_pref_data)}")
if missing_files:
    print(f"  Missing: {missing_files[:10]}{'...' if len(missing_files) > 10 else ''}")

✓ Loaded 5/5 files
  Total samples: 105


In [11]:
display(full_pref_data.head(2))
messages_sample = full_pref_data.loc[0, "messages"]
print("Sample messages:")
print(messages_sample[-1])

,conversation,messages,winning_response,losing_response,winning_scores_list,losing_scores_list,winning_scores_avg_list,losing_scores_avg_list,winning_score_final,losing_score_final,winning_conversation,losing_conversation
0,"['Hello, welcome to your first motivational se...","[{'role': 'system', 'content': 'You are a moti...",It seems that you believe that some people cou...,"I see, James... I think you`ve come to the rig...","[[2, 2, 2, 2, 1], [3, 2, 2, 3, 3, 2, 4, 4, 3, ...","[[2, 2, 1, 2, 1], [3, 2, 2, 2, 2, 2, 3, 3, 3, ...","[np.float64(1.8), np.float64(2.8823529411764706)]","[np.float64(1.6), np.float64(2.4705882352941178)]",2.341176,2.035294,"['Hello, welcome to your first motivational se...","['Hello, welcome to your first motivational se..."
1,"['Hello, welcome to your first motivational se...","[{'role': 'system', 'content': 'You are a moti...",I respect that. Let me help you by finding ano...,This session is about learning what motivates ...,"[[2, 2, 1, 2, 2], [3, 2, 2, 3, 3, 2, 4, 3, 3, ...","[[2, 2, 1, 1, 1], [2, 2, 1, 2, 2, 2, 3, 2, 2, ...","[np.float64(1.8), np.float64(2.764705882352941)]","[np.float64(1.4), np.float64(1.9411764705882353)]",2.282353,1.670588,"['Hello, welcome to your first motivational se...","['Hello, welcome to your first motivational se..."


Sample messages:
{'role': 'user', 'content': "Yeah, I'm James, 27. Honestly, I don't really believe in this therapy stuff. I'm here because someone thought it would be a good idea for me to come, not because I wanted to. I've picked up smoking over the last few months and it's become part of my daily routine. I'm starting to worry about how it's affecting my health, but I've never actually tried to quit or anything. So, what do you want to know?"}


In [12]:
# ─────────────────────────────────────────────────────────────────────────────────
# Filter & Format Data
# ─────────────────────────────────────────────────────────────────────────────────

# Filter by minimum messages length
initial_count = len(full_pref_data)
full_pref_data = full_pref_data[
    full_pref_data["messages"].apply(len) >= MIN_MESSAGES_LENGTH
]
print(f"✓ Filtered by messages length: {len(full_pref_data)}/{initial_count} samples (min={MIN_MESSAGES_LENGTH})")

# Filter by score threshold
pref_data = full_pref_data[
    full_pref_data["winning_score_final"] - full_pref_data["losing_score_final"] >= SCORE_THRESHOLD
].copy()
print(f"✓ Filtered by score threshold: {len(pref_data)}/{len(full_pref_data)} samples (threshold={SCORE_THRESHOLD})")

# Select and rename columns
pref_data = pref_data[["messages", "winning_response", "losing_response", "winning_score_final", "losing_score_final"]]
pref_data = pref_data.rename(columns={"winning_response": "chosen", "losing_response": "rejected"})

# Apply chat template to create prompts
pref_data["prompt"] = pref_data["messages"].apply(
    lambda x: tokenizer.apply_chat_template(x, add_generation_prompt=True, tokenize=False)
)

# ─────────────────────────────────────────────────────────────────────────────────
# Create Transcript (plain text with [PATIENT]/[THERAPIST] labels for oracle eval)
# ─────────────────────────────────────────────────────────────────────────────────
def format_conversation_for_oracle(messages):
    """
    Converts a list of message dicts into a plain text transcript 
    using [PATIENT] and [THERAPIST] labels. 
    Removes the System Prompt entirely.
    """
    transcript = []
    
    for msg in messages:
        role = msg.get("role")
        content = msg.get("content", "").strip()
        
        if not content:
            continue
            
        if role == "user":
            transcript.append(f"[PATIENT]: {content}")
        elif role == "assistant":
            transcript.append(f"[THERAPIST]: {content}")
        # Explicitly ignore 'system' role
        
    return "\n\n".join(transcript)

pref_data["transcript"] = pref_data["messages"].apply(format_conversation_for_oracle)
print(f"✓ Transcripts created (plain text with [PATIENT]/[THERAPIST] labels)")

# Sanity check
print("\n--- Sample Transcript ---")
sample_transcript = pref_data["transcript"].iloc[0]
print(sample_transcript[:500] if len(sample_transcript) > 500 else sample_transcript)
print("-------------------------")

# Ensure responses end with stop string
def ensure_stop_string(text):
    if not text.strip().endswith(STOP_STRING_PRIMARY):
        return f"{text} {STOP_STRING_PRIMARY}{tokenizer.eos_token}"
    return text

pref_data["chosen"] = pref_data["chosen"].apply(ensure_stop_string)
pref_data["rejected"] = pref_data["rejected"].apply(ensure_stop_string)

print(f"✓ Prompts formatted with chat template")

✓ Filtered by messages length: 105/105 samples (min=0)
✓ Filtered by score threshold: 68/105 samples (threshold=0.1)
✓ Transcripts created (plain text with [PATIENT]/[THERAPIST] labels)

--- Sample Transcript ---
[THERAPIST]: Hello, welcome to your first motivational session with me. My name is David and I`m a professional motivational counselor. Can you start by telling me a little bit about yourself and why are you here?

[PATIENT]: Yeah, I'm James, 27. Honestly, I don't really believe in this therapy stuff. I'm here because someone thought it would be a good idea for me to come, not because I wanted to. I've picked up smoking over the last few months and it's become part of my daily routine. I'm start
-------------------------
✓ Prompts formatted with chat template


In [13]:
# ─────────────────────────────────────────────────────────────────────────────────
# Handle Prompt Length
# ─────────────────────────────────────────────────────────────────────────────────
prompt_token_lengths = pref_data["prompt"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=False))
)
max_length = prompt_token_lengths.max()
exceeds_limit = (prompt_token_lengths > MAX_ALLOWED_PROMPT_LENGTH).sum()

print(f"Prompt length stats:")
print(f"  Max: {max_length} tokens | Limit: {MAX_ALLOWED_PROMPT_LENGTH}")
print(f"  Exceeding limit: {exceeds_limit}/{len(pref_data)}")

# Truncate if needed (keep the end for recent context)
if max_length > MAX_ALLOWED_PROMPT_LENGTH:
    print(f"  Truncating to {MAX_ALLOWED_PROMPT_LENGTH} tokens...")
    
    def truncate_prompt(row):
        if prompt_token_lengths.loc[row.name] <= MAX_ALLOWED_PROMPT_LENGTH:
            return row["prompt"]
        tokens = tokenizer.encode(row["prompt"], add_special_tokens=False)
        return tokenizer.decode(tokens[-MAX_ALLOWED_PROMPT_LENGTH:], skip_special_tokens=False)
    
    pref_data["prompt"] = pref_data.apply(truncate_prompt, axis=1)
    new_max = max(len(tokenizer.encode(p, add_special_tokens=False)) for p in pref_data["prompt"])
    print(f"  ✓ Max length after truncation: {new_max} tokens")
else:
    print("  ✓ All prompts within limit")

Prompt length stats:
  Max: 5196 tokens | Limit: 4096
  Exceeding limit: 3/68
  Truncating to 4096 tokens...
  ✓ Max length after truncation: 4096 tokens


In [14]:
# ─────────────────────────────────────────────────────────────────────────────────
# Create HuggingFace Dataset
# ─────────────────────────────────────────────────────────────────────────────────
dataset_dict = {
    "prompt": pref_data["prompt"].tolist(),
    "transcript": pref_data["transcript"].tolist(),
    "chosen": pref_data["chosen"].tolist(),
    "rejected": pref_data["rejected"].tolist(),
}
full_dataset = Dataset.from_dict(dataset_dict)

# Train/validation split
train_eval_split = full_dataset.train_test_split(test_size=EVAL_SPLIT_RATIO, seed=SEED, shuffle=True)
train_dataset = train_eval_split["train"]
eval_dataset = train_eval_split["test"]

# Validation
assert len(train_dataset) > 0, "Train dataset is empty!"
assert len(eval_dataset) > 0, "Eval dataset is empty!"

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"  Train samples: {len(train_dataset)}")
print(f"  Eval samples:  {len(eval_dataset)}")
print(f"  Split ratio:   {EVAL_SPLIT_RATIO}")
print(f"  Columns:       {list(full_dataset.column_names)}")
print("=" * 50)

DATASET SUMMARY
  Train samples: 64
  Eval samples:  4
  Split ratio:   0.05
  Columns:       ['prompt', 'transcript', 'chosen', 'rejected']


---
## 6. Quantization & LoRA Configuration

Defines 4-bit quantization and LoRA adapter settings used by the GRPO trainer.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────────
# 4-bit Quantization Config
# ─────────────────────────────────────────────────────────────────────────────────
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=(
        torch.bfloat16 
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported() 
        else torch.float16
    ),
)

# ─────────────────────────────────────────────────────────────────────────────────
# LoRA Config (Policy - Causal LM)
# ─────────────────────────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

print("✓ Quantization and LoRA configs ready")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  Target modules: {LORA_TARGET_MODULES}")

✓ Quantization and LoRA configs ready
  LoRA r=16, alpha=16, dropout=0.05
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj']


---
## 7. Oracle Reward Model

Async GPT-4o-mini reward function with **JSON-schema validation**, concurrency limits, and retries.
Scores are aggregated across questionnaires into a mean reward.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         ORACLE REWARD FUNCTIONS                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

_oracle_sem = asyncio.Semaphore(ORACLE_MAX_CONCURRENCY)

async def get_evaluation_json(
    transcript: str,
    completion: str,
    questionnaire_id: int,
    max_retries: int = ORACLE_MAX_RETRIES,
    eval_temperature: float = EVAL_TEMPERATURE,
):
    """
    Get questionnaire evaluation scores from oracle model.
    
    Args:
        transcript: Plain text conversation transcript ([PATIENT]/[THERAPIST] format).
        completion: Model-generated completion to append to transcript.
        questionnaire_id: Which questionnaire to use for evaluation.
    
    Returns:
        Tuple of (data_dict, n_questions). data_dict is None on failure.
    """
    full_conversation = f"{transcript}\n\n[THERAPIST]: {completion}"
    eval_dict = get_prompt_eval_questionnaire(
        questionnaire=questionnaire_id,
        conversation=full_conversation
    )
    eval_prompt = eval_dict["prompt"]
    n_questions = int(eval_dict["questions_count"])
    schema = eval_dict["schema"]

    for attempt in range(max_retries):
        try:
            async with _oracle_sem:
                resp = await client.chat.completions.create(
                    model=ORACLE_MODEL_ID,
                    messages=[{"role": "user", "content": eval_prompt}],
                    temperature=eval_temperature,
                    max_tokens=256,
                    response_format={
                        "type": "json_schema",
                        "json_schema": {
                            "name": f"questionnaire_{questionnaire_id}_evaluation",
                            "schema": schema,
                            "strict": True,
                        },
                    },
                )
            content = resp.choices[0].message.content
            if not content or not content.strip():
                raise ValueError("Empty oracle response")

            data = json.loads(content)

            # Validate response
            if data.get("questionnaire_id") != questionnaire_id:
                raise ValueError(f"Wrong questionnaire_id: {data.get('questionnaire_id')}")
            
            scores = data.get("scores", [])
            if len(scores) != n_questions:
                raise ValueError(f"Expected {n_questions} scores, got {len(scores)}")
            if any(not isinstance(s, int) or s < 1 or s > 5 for s in scores):
                raise ValueError("Invalid score values")

            data["mean_score"] = float(np.mean(scores))
            return data, n_questions

        except Exception as e:
            if attempt >= max_retries - 1:
                print(f"  ⚠ Oracle failed (qid={questionnaire_id}): {e}")
                return None, n_questions
            await asyncio.sleep(2 ** attempt)  # Exponential backoff

    return None, n_questions


async def process_single_sample(idx: int, transcript: str, completion: str, questionnaire_ids=QUESTIONNAIRE_IDS):
    """Compute reward for one sample by averaging scores across questionnaires.
    
    Args:
        idx: Sample index for logging.
        transcript: Plain text conversation transcript ([PATIENT]/[THERAPIST] format).
        completion: Model-generated completion.
        questionnaire_ids: List of questionnaire IDs to evaluate.
    """
    rewards = []
    
    for qid in questionnaire_ids:
        data, _ = await get_evaluation_json(
            transcript=transcript,
            completion=completion,
            questionnaire_id=int(qid),
            eval_temperature=EVAL_TEMPERATURE,
        )
        rewards.append(float(data["mean_score"]) if data else 0.0)

    if (idx + 1) % 10 == 0:
        print(f"    Evaluated sample {idx + 1}")

    return float(np.mean(rewards)) if rewards else 0.0


async def _reward_func_async(prompts, completions, transcript, questionnaire_ids=QUESTIONNAIRE_IDS, **kwargs):
    """Async batch reward computation.
    
    Args:
        prompts: Original prompts (unused, kept for GRPOTrainer compatibility).
        completions: Model-generated completions.
        transcript: List of plain text transcripts for oracle evaluation.
        questionnaire_ids: List of questionnaire IDs to evaluate.
    """
    tasks = [
        process_single_sample(idx, t, c, questionnaire_ids)
        for idx, (t, c) in enumerate(zip(transcript, completions))
    ]
    return await asyncio.gather(*tasks)


def reward_client_func(prompts, completions, transcript, questionnaire_ids=QUESTIONNAIRE_IDS, **kwargs):
    """Synchronous wrapper for TRL GRPOTrainer compatibility.
    
    Args:
        prompts: Original prompts (unused, kept for GRPOTrainer compatibility).
        completions: Model-generated completions.
        transcript: List of plain text transcripts for oracle evaluation.
        questionnaire_ids: List of questionnaire IDs to evaluate.
    """
    coro = _reward_func_async(prompts, completions, transcript, questionnaire_ids=questionnaire_ids, **kwargs)
    try:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(coro)
    except RuntimeError:
        return asyncio.run(coro)


# Exposed for GRPOTrainer
reward_funcs = reward_client_func
reward_processing_classes = None

print("✓ Oracle reward functions defined")
print(f"  Questionnaires: {QUESTIONNAIRE_IDS}")
print(f"  Max concurrency: {ORACLE_MAX_CONCURRENCY}")
print(f"  Using 'transcript' column for oracle evaluation")

✓ Oracle reward functions defined
  Questionnaires: [1, 2]
  Max concurrency: 8
  Using 'transcript' column for oracle evaluation


In [17]:
# ─────────────────────────────────────────────────────────────────────────────────
# Quick unit-style test for get_evaluation_json
# ─────────────────────────────────────────────────────────────────────────────────
_test_transcript = train_dataset[0]["transcript"]
# print("\n--- Test Transcript ---")
# print(_test_transcript)
# print("-----------------------")
_test_completion = "It makes sense to feel that way. What situations bring it up most?"

data, nq = await get_evaluation_json(
    transcript=_test_transcript,
    completion=_test_completion,
    questionnaire_id=QUESTIONNAIRE_IDS[1],
)

print(f"Questions: {nq}")
print(f"Result: {data}")

Questions: 17
Result: {'questionnaire_id': 2, 'scores': [2, 2, 2, 3, 2, 2, 3, 3, 2, 2, 3, 3, 2, 2, 2, 2, 2], 'mean_score': 2.2941176470588234}


In [18]:
# # ─────────────────────────────────────────────────────────────────────────────────
# # Test Oracle Reward (optional sanity check)
# # ─────────────────────────────────────────────────────────────────────────────────
# sample_prompt = train_dataset[0]["prompt"]
# sample_transcript = train_dataset[0]["transcript"]
# sample_completion_chosen = train_dataset[0]["chosen"]

# print("Testing oracle reward...")
# print(f"  Original prompt length: {len(sample_prompt)}")
# print(f"  Transcript length:      {len(sample_transcript)}")

# data, n = await get_evaluation_json(sample_transcript, sample_completion_chosen, questionnaire_id=1)
# print(f"  Single questionnaire (Q1): {data}")

# rewards = reward_client_func(
#     [sample_prompt], 
#     [sample_completion_chosen], 
#     transcript=[sample_transcript],
#     questionnaire_ids=QUESTIONNAIRE_IDS
# )
# print(f"  Combined reward (Q{QUESTIONNAIRE_IDS}): {rewards[0]:.3f}")
# print("✓ Oracle test passed")

---
## 8. Load Policy Model

Always loads the base model, then (in training mode) detects the latest **valid** checkpoint to resume from.

In [19]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                           LOAD POLICY MODEL                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────────
# Resolve Local Output Directory (EXPERIMENT_NAME already includes suffix)
# ─────────────────────────────────────────────────────────────────────────────────
if IN_COLAB:
    LOCAL_OUTDIR = f"/content/drive/MyDrive/grpo_runs/{EXPERIMENT_NAME}"
else:
    LOCAL_OUTDIR = f"./grpo_runs/{EXPERIMENT_NAME}"

os.makedirs(LOCAL_OUTDIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────────
# Detect checkpoint state
# ─────────────────────────────────────────────────────────────────────────────────
available_checkpoints = list_local_checkpoints(LOCAL_OUTDIR)
latest_checkpoint = get_latest_checkpoint(LOCAL_OUTDIR)
resume_from_checkpoint = None  # Will be set for training resume

print("=" * 60)
print(f"Mode: {'🔍 EXPLORATION' if EXPLORATION_MODE else '🚀 TRAINING'}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Output dir: {LOCAL_OUTDIR}")
print(f"Checkpoints found: {len(available_checkpoints)}")
if available_checkpoints:
    print(f"  Available: {[os.path.basename(c) for c in available_checkpoints]}")
print("=" * 60)

# ─────────────────────────────────────────────────────────────────────────────────
# Load base model (always)
# ─────────────────────────────────────────────────────────────────────────────────
base_policy = load_base_policy(quantization_config=bnb_4bit)
policy = base_policy
patch_generate(policy)


if TRAINING_MODE:
    # ═══════════════════════════════════════════════════════════════════════════
    # TRAINING MODE: trainer will handle adapter + resume
    # ═══════════════════════════════════════════════════════════════════════════
    if latest_checkpoint:
        # Validate before setting resume path
        is_valid, missing = validate_checkpoint_integrity(latest_checkpoint)
        if is_valid:
            resume_from_checkpoint = latest_checkpoint
            print(f"Will resume from: {os.path.basename(latest_checkpoint)}")
        else:
            print(f"⚠ Latest checkpoint invalid (missing: {missing}), starting fresh")
    else:
        print("No existing checkpoints — starting fresh")
    

# ─────────────────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────────────────
param_info = get_adapter_param_count(policy)
print(f"✓ Policy loaded ({param_info['trainable_M']:.2f}M trainable)")
print("=" * 60)

Mode: 🔍 EXPLORATION
Experiment: GRPO_Oracle_Llama32-1B-Instruct_LA5_G4_V2
Output dir: ./grpo_runs/GRPO_Oracle_Llama32-1B-Instruct_LA5_G4_V2
Checkpoints found: 15
  Available: ['checkpoint-496', 'checkpoint-992', 'checkpoint-1488', 'checkpoint-1984', 'checkpoint-2480', 'checkpoint-2976', 'checkpoint-3472', 'checkpoint-3968', 'checkpoint-4464', 'checkpoint-4960', 'checkpoint-5456', 'checkpoint-5952', 'checkpoint-6448', 'checkpoint-6944', 'checkpoint-7440']
  Loading base model with 4-bit quantization...
  ✓ Patched generate() for LlamaForCausalLM
✓ Policy loaded (0.00M trainable)


---
## 9. GRPO Training

### 9.1 Trainer Initialization (Training Mode Only)

In **Training Mode** (`EXPLORATION_MODE = False`), the trainer is initialized and will auto-resume from the latest valid checkpoint if available.
W&B runs use consistent IDs for resumption, and each checkpoint writes `experiment_metadata.json`.

In **Exploration Mode** (`EXPLORATION_MODE = True`), the trainer is NOT initialized — the model is loaded in read-only mode for evaluation.

In [20]:
# ─────────────────────────────────────────────────────────────────────────────────
# W&B Run Initialization (consistent naming + resume support)
# ─────────────────────────────────────────────────────────────────────────────────
if REPORT_TO == "wandb" and TRAINING_MODE:
    # Use EXPERIMENT_NAME directly (already includes _QUICKTEST suffix if applicable)
    wandb_run_id = EXPERIMENT_NAME
    wandb_run_name = EXPERIMENT_NAME
    wandb_project = "GRPO_Experiments"
    
    wandb_config = {
        # Experiment identifiers
        "experiment_name": EXPERIMENT_NAME,
        "experiment_name_base": EXPERIMENT_NAME_BASE,
        "quick_test_mode": QUICK_TEST_MODE,
        "base_model": BASE_MODEL_ID,
        "oracle_model": ORACLE_MODEL_ID,
        "adapter_repo": CURRENT_ADAPTER_REPO,
        # Data
        "data_path": DATA_SUBPATH,
        "num_data_files": NUM_DATA_FILES,
        "score_threshold": SCORE_THRESHOLD,
        "train_samples": len(train_dataset),
        "eval_samples": len(eval_dataset),
        # Training hyperparameters
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_TRAIN_EPOCHS,
        "num_generations": NUM_GENERATIONS,
        "grpo_beta": GRPO_BETA,
        "grpo_temperature": GRPO_TEMPERATURE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "max_completion_length": MAX_COMPLETION_LENGTH,
        # Oracle / Reward
        "questionnaire_ids": QUESTIONNAIRE_IDS,
        # LoRA
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        # Mode
        "mode": "training",
    }
    
    # Handle notebook re-run: check if a run is already active
    if wandb.run is not None:
        if wandb.run.id == wandb_run_id:
            print(f"✓ W&B run already active with correct ID: {wandb_run_id}")
        else:
            print(f"⚠ Different W&B run active ({wandb.run.id}), finishing...")
            wandb.finish()
            wandb.init(
                project=wandb_project,
                name=wandb_run_name,
                id=wandb_run_id,
                resume="allow",
                config=wandb_config,
            )
            print(f"✓ W&B run initialized: {wandb_run_name} (project={wandb_project})")
    else:
        wandb.init(
            project=wandb_project,
            name=wandb_run_name,
            id=wandb_run_id,
            resume="allow",
            config=wandb_config,
        )
        print(f"✓ W&B run initialized: {wandb_run_name} (project={wandb_project})")
elif EXPLORATION_MODE:
    print("🔍 EXPLORATION MODE: W&B logging disabled (read-only session)")
    REPORT_TO = "none"  # Override to prevent logging during exploration
else:
    print("(W&B logging disabled — skipping wandb.init())")

🔍 EXPLORATION MODE: W&B logging disabled (read-only session)


In [21]:
# ─────────────────────────────────────────────────────────────────────────────────
# GRPO Config & Trainer Initialization (Training Mode Only)
# ─────────────────────────────────────────────────────────────────────────────────

if TRAINING_MODE:
    grpo_args = GRPOConfig(
        # Output & Hub
        output_dir=LOCAL_OUTDIR,
        hub_model_id=CURRENT_ADAPTER_REPO,
        
        # Batch sizes
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        
        # Training
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        num_iterations=1,
        num_generations=NUM_GENERATIONS,
        max_completion_length=MAX_COMPLETION_LENGTH,
        beta=GRPO_BETA,
        temperature=GRPO_TEMPERATURE,
        remove_unused_columns=False,
        
        # Cyclic Learning Rate Schedule (cosine with restarts)
        lr_scheduler_type="cosine_with_restarts",
        lr_scheduler_kwargs={"num_cycles": NUM_CYCLES},
        warmup_ratio=WARMUP_RATIO,
        
        # Logging
        logging_steps=LOGGING_STEPS,
        report_to=REPORT_TO,
        log_completions=LOG_COMPLETIONS,
        
        # Checkpointing — save every epoch, keep ALL checkpoints
        save_strategy=SAVE_STRATEGY,
        save_total_limit=SAVE_TOTAL_LIMIT,
        push_to_hub=PUSH_TO_HUB,
        hub_strategy="all_checkpoints",
    )

    # ─────────────────────────────────────────────────────────────────────────────
    # Checkpoint Metadata Callback
    # ─────────────────────────────────────────────────────────────────────────────
    from transformers import TrainerCallback
    
    class CheckpointMetadataCallback(TrainerCallback):
        """Save experiment metadata alongside each checkpoint."""
        
        def on_save(self, args, state, control, **kwargs):
            checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
            if not os.path.isdir(checkpoint_dir):
                return
            
            metadata = {
                "experiment_name": EXPERIMENT_NAME,
                "base_model": BASE_MODEL_ID,
                "oracle_model": ORACLE_MODEL_ID,
                "global_step": state.global_step,
                "epoch": state.epoch,
                "best_metric": state.best_metric,
                "questionnaire_ids": QUESTIONNAIRE_IDS,
                "grpo_beta": GRPO_BETA,
                "learning_rate": LEARNING_RATE,
                "lora_r": LORA_R,
                "timestamp": pd.Timestamp.now().isoformat(),
            }
            
            with open(os.path.join(checkpoint_dir, "experiment_metadata.json"), "w") as f:
                json.dump(metadata, f, indent=2)

    # ─────────────────────────────────────────────────────────────────────────────
    # Initialize Trainer
    # Trainer creates new LoRA adapter via peft_config (fresh) OR loads from 
    # resume_from_checkpoint (resume) — we pass base model either way
    # ─────────────────────────────────────────────────────────────────────────────
    trainer = GRPOTrainer(
        model=policy,
        args=grpo_args,
        processing_class=tokenizer,
        reward_funcs=reward_funcs,
        reward_processing_classes=reward_processing_classes,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=lora_config,  # Trainer uses this for fresh start, ignores on resume
        callbacks=[CheckpointMetadataCallback()],
    )

    patch_generate(trainer.model)
    
    param_info = get_adapter_param_count(trainer.model)
    print("=" * 60)
    print("TRAINER INITIALIZED")
    print("=" * 60)
    print(f"  Epochs:         {NUM_TRAIN_EPOCHS}")
    print(f"  Batch size:     {TRAIN_BATCH_SIZE}")
    print(f"  Learning rate:  {LEARNING_RATE}")
    print(f"  Resume from:    {os.path.basename(resume_from_checkpoint) if resume_from_checkpoint else 'None (fresh)'}")
    print(f"  Trainable:      {param_info['trainable_M']:.2f}M params")
    print("=" * 60)

else:
    trainer = None
    print("=" * 60)
    print("🔍 EXPLORATION MODE — Trainer not initialized")
    print("   Proceed to evaluation section.")
    print("=" * 60)

🔍 EXPLORATION MODE — Trainer not initialized
   Proceed to evaluation section.


### 9.2 Training Loop (Training Mode Only)

- **Training Mode**: Runs the GRPO training loop with auto-resume support.
- **Exploration Mode**: Skips training entirely and proceeds to evaluation.

In [22]:
# ─────────────────────────────────────────────────────────────────────────────────
# Training Loop (Training Mode Only)
# ─────────────────────────────────────────────────────────────────────────────────

if TRAINING_MODE:
    print("=" * 60)
    print(f"STARTING GRPO TRAINING — {EXPERIMENT_NAME}")
    print("=" * 60)
    
    if resume_from_checkpoint:
        print(f"[RESUME] From: {os.path.basename(resume_from_checkpoint)}")
    else:
        print("[START] Fresh training run")
    
    # Train with optional resume — trainer handles adapter loading internally
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

else:
    print("=" * 60)
    print("🔍 EXPLORATION MODE — Training Skipped")
    print("=" * 60)
    print("  → Proceed to Section 10 for evaluation")
    print("=" * 60)

🔍 EXPLORATION MODE — Training Skipped
  → Proceed to Section 10 for evaluation


In [23]:
# ─────────────────────────────────────────────────────────────────────────────────
# Save & Push Final Adapter (Training Mode Only)
# ─────────────────────────────────────────────────────────────────────────────────

if TRAINING_MODE:
    print("-" * 60)
    print("Saving final adapter...")

    final_adapter_path = f"{LOCAL_OUTDIR}/final_adapter"
    trainer.model.save_pretrained(final_adapter_path)
    print(f"  ✓ Saved to: {final_adapter_path}")

    if PUSH_TO_HUB:
        trainer.model.push_to_hub(
            repo_id=CURRENT_ADAPTER_REPO,
            commit_message="Final adapter push"
        )
        print(f"  ✓ Pushed to Hub: {CURRENT_ADAPTER_REPO}")

    # Summary
    print("=" * 60)
    print("TRAINING COMPLETE")
    print("=" * 60)
    print(f"  Experiment:  {EXPERIMENT_NAME}")
    print(f"  Hub repo:    {CURRENT_ADAPTER_REPO}")
    print(f"  Local dir:   {LOCAL_OUTDIR}")
    print(f"  Checkpoints: {len(list_local_checkpoints(LOCAL_OUTDIR))}")
    print("=" * 60)
    print("\nTo run the exploration sweep, set:")
    print("  EXPLORATION_MODE = True")
else:
    print("🔍 EXPLORATION MODE — Save step skipped")
    print("   Proceed to generation/evaluation tests below.")

🔍 EXPLORATION MODE — Save step skipped
   Proceed to generation/evaluation tests below.


---
## 10. Exploration Helpers

Utilities for **multi-adapter** evaluation sweeps. Checkpoints are loaded once and switched with `set_adapter()` / `disable_adapter()` using 0-indexed epoch lists.

In [24]:
# ─────────────────────────────────────────────────────────────────────────────────
# Exploration Sweep Helpers (Multi-Adapter Approach)
# ─────────────────────────────────────────────────────────────────────────────────

def _generate_single(model, prompt, max_new_tokens=50, temperature=DEFAULT_TEMPERATURE):
    """Generate a single completion from a prompt and return decoded text."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        temperature=temperature,
        tokenizer=tokenizer,
    )
    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=False
    )


def validate_sweep_epochs(epochs: list[int]) -> list[int]:
    """
    Validate a list of epochs for exploration sweep (0-indexed, 0 = base model).
    
    Args:
        epochs: List of epoch indices to sweep (0 = base model).
    
    Returns:
        The validated list of epoch indices.
    """
    if not isinstance(epochs, list) or len(epochs) == 0:
        raise ValueError("epochs must be a non-empty list")
    if any(not isinstance(e, int) for e in epochs):
        raise ValueError("epochs must contain only integers")
    if any(e < 0 for e in epochs):
        raise ValueError("epochs must be >= 0")
    return epochs


def build_multi_adapter_model(base_policy, output_dir: str, epochs: list[int]) -> tuple:
    """
    Build a single PeftModel with multiple adapters loaded for efficient sweeping.
    
    This approach wraps base_policy ONCE and loads each checkpoint as a named adapter,
    allowing runtime switching via set_adapter() / disable_adapter().
    
    NOTE: This call MUTATES base_policy in-place by attaching adapter modules.
    If you need a clean base model afterwards, reload it or pass a fresh instance.
    
    Args:
        base_policy: The base model (no adapter attached).
        output_dir: Directory containing checkpoints.
        epochs: List of 0-indexed epoch numbers (0 = base, 1+ = checkpoint epochs).
    
    Returns:
        Tuple of (multi_adapter_model, labels_by_epoch) where:
        - multi_adapter_model: A PeftModel with all adapters loaded (or base_policy if only epoch 0)
        - labels_by_epoch: Dict mapping epoch -> label string (e.g., {0: "base", 3: "epoch_3"})
    """
    labels_by_epoch = {}
    non_zero_epochs = [e for e in epochs if e != 0]
    
    # Build labels for all epochs
    for epoch in epochs:
        labels_by_epoch[epoch] = "base" if epoch == 0 else f"epoch_{epoch}"
    
    # If only base model requested, return as-is
    if not non_zero_epochs:
        print("  Only base model requested — no adapters to load")
        patch_generate(base_policy)
        base_policy.eval()
        return base_policy, labels_by_epoch
    
    # Find checkpoint paths for all non-zero epochs
    epoch_to_path = {}
    for epoch in non_zero_epochs:
        ckpt_path = get_checkpoint_by_identifier(output_dir, epoch)
        if ckpt_path is None:
            available = [os.path.basename(c) for c in list_local_checkpoints(output_dir)]
            raise FileNotFoundError(
                f"Checkpoint for epoch {epoch} not found! Available: {available}"
            )
        is_valid, missing = validate_checkpoint_integrity(ckpt_path)
        if not is_valid:
            raise RuntimeError(f"Checkpoint {epoch} incomplete! Missing: {missing}")
        epoch_to_path[epoch] = ckpt_path
    
    # Create PeftModel from FIRST non-zero epoch
    first_epoch = non_zero_epochs[0]
    first_path = epoch_to_path[first_epoch]
    first_adapter_name = labels_by_epoch[first_epoch]
    
    print(f"  Creating PeftModel from epoch {first_epoch} (adapter: '{first_adapter_name}')")
    multi_model = PeftModel.from_pretrained(
        base_policy,
        first_path,
        adapter_name=first_adapter_name,
        is_trainable=False,
    )
    
    # Load remaining adapters
    for epoch in non_zero_epochs[1:]:
        ckpt_path = epoch_to_path[epoch]
        adapter_name = labels_by_epoch[epoch]
        print(f"  Loading adapter for epoch {epoch} (adapter: '{adapter_name}')")
        multi_model.load_adapter(ckpt_path, adapter_name=adapter_name, is_trainable=False)
    
    patch_generate(multi_model)
    multi_model.eval()
    
    print(f"  ✓ Multi-adapter model ready with {len(non_zero_epochs)} adapter(s)")
    if hasattr(multi_model, "peft_config"):
        print(f"    Available adapters: {list(multi_model.peft_config.keys())}")
    else:
        print("    Available adapters: <unknown>")
    
    return multi_model, labels_by_epoch


def run_exploration_sweep(
    prompts,
    base_policy,
    output_dir: str,
    epochs: list[int],
    max_new_tokens: int = 50,
    temperature: float = DEFAULT_TEMPERATURE,
    truncate_prompt_print: int = 128,
    questionnaire_ids = [],
    ):
    """
    Run sweep across checkpoints using a single multi-adapter model.
    
    Uses set_adapter() / disable_adapter() for efficient epoch switching
    instead of loading separate model instances.
    
    Args:
        prompts: List of prompts to evaluate.
        base_policy: The base model (no adapter attached).
        output_dir: Directory containing checkpoints.
        epochs: List of 0-indexed epoch indices to sweep (0 = base model).
        max_new_tokens: Max tokens to generate per completion.
        temperature: Sampling temperature.
        truncate_prompt_print: Truncate prompt display to this many chars (None = no truncation).
    """
    sweep_epochs = validate_sweep_epochs(epochs)
    
    print("=" * 80)
    print("EXPLORATION SWEEP (Multi-Adapter)")
    print("=" * 80)
    print(f"  Epochs (0-indexed): {sweep_epochs}")
    print(f"  0 = base model (adapters disabled), 1+ = adapter epochs")
    print(f"  Prompts: {len(prompts)}")
    print(f"  Max tokens: {max_new_tokens} | Temperature: {temperature}")
    print("=" * 80)
    
    # Build single multi-adapter model
    print("\nBuilding multi-adapter model...")
    multi_model, labels_by_epoch = build_multi_adapter_model(base_policy, output_dir, sweep_epochs)
    
    # Check if we have adapter switching available
    has_adapters = hasattr(multi_model, "set_adapter")
    print(f"✓ Model ready (adapters enabled: {has_adapters})\n")
    
    # Generate outputs grouped by prompt
    for i, prompt in enumerate(prompts, start=1):
        print("\n" + "=" * 80)
        print(f"[Prompt {i}]")
        if truncate_prompt_print is not None and len(prompt) > truncate_prompt_print:
            print(f"Input(truncated): ...{prompt[-truncate_prompt_print:]}")
        else:
            print(f"Input: {prompt}")
        print("-" * 80)
        
        for epoch in sweep_epochs:
            label = labels_by_epoch[epoch]
            
            # Switch adapter or disable for base
            if has_adapters:
                if epoch == 0:
                    # Disable all adapters to get base model behavior
                    with multi_model.disable_adapter():
                        response = _generate_single(
                            multi_model,
                            prompt,
                            max_new_tokens=max_new_tokens,
                            temperature=temperature,
                        )
                else:
                    # Activate the specific adapter
                    multi_model.set_adapter(label)
                    multi_model.eval()
                    response = _generate_single(
                        multi_model,
                        prompt,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature,
                    )
            else:
                # No adapters loaded, just generate from base
                response = _generate_single(
                    multi_model,
                    prompt,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                )
            
            print(f"[{label}]:")
            print(response)
            print("-" * 40)
            
        # Clear cache after finishing all versions of one prompt
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print("\n" + "=" * 80)
    print("SWEEP COMPLETE")
    print("=" * 80)



print("✓ Model helper functions defined")

✓ Model helper functions defined


---
## 11. Post-Training Evaluation

Deterministically samples test prompts from the training set and runs the exploration sweep configuration below.

In [25]:
# ─────────────────────────────────────────────────────────────────────────────────
# Test prompts (sampled from training data, deterministic)
# ─────────────────────────────────────────────────────────────────────────────────
NUM_TEST_PROMPTS = 2
_rng = random.Random(0)
_sample_indices = _rng.sample(range(len(train_dataset)), k=min(NUM_TEST_PROMPTS, len(train_dataset)))
TEST_PROMPTS = [train_dataset[i]["prompt"] for i in _sample_indices]

print("Test prompt sampling")
print(f"  Seed:      {SEED}")
print(f"  Indices:   {_sample_indices}")
print(f"  Count:     {len(TEST_PROMPTS)}")

# for i, idx in enumerate(_sample_indices):
#     print(f"\n--- Test Prompt {i + 1} (Index {idx}) ---")
#     print(("..." if len(TEST_PROMPTS[i]) > 500 else "") + TEST_PROMPTS[i][-500:])

Test prompt sampling
  Seed:      42
  Indices:   [49, 53]
  Count:     2


In [27]:
# ─────────────────────────────────────────────────────────────────────────────────
# Exploration Sweep (Exploration Mode Only)
# ─────────────────────────────────────────────────────────────────────────────────
# 0-indexed epochs: 0 = base model, 1 = first checkpoint, 2 = second checkpoint, ...

#
start_epoch = 0
end_epoch = len(available_checkpoints) # 15
jump = 3

SWEEP_EPOCHS = list(range(start_epoch, end_epoch + 1, jump))

# SWEEP_EPOCHS = [0, 2, 4, 7, 10, 11
print(f"Exploration sweep epochs: {SWEEP_EPOCHS}")

# multi_adapter_model, adapters_labels = build_multi_adapter_model(base_policy, LOCAL_OUTDIR, SWEEP_EPOCHS)

# Run exploration sweep (base_policy will have adapters loaded but disabled)
if EXPLORATION_MODE and True:
    run_exploration_sweep(
        prompts=TEST_PROMPTS,
        base_policy=base_policy,
        output_dir=LOCAL_OUTDIR,
        epochs=SWEEP_EPOCHS,
        max_new_tokens=200,
        temperature=0.9,
        truncate_prompt_print=512,
    )
else:
    print("Exploration sweep skipped (Training Mode active).")

Exploration sweep epochs: [0, 3, 6, 9, 12, 15]
EXPLORATION SWEEP (Multi-Adapter)
  Epochs (0-indexed): [0, 3, 6, 9, 12, 15]
  0 = base model (adapters disabled), 1+ = adapter epochs
  Prompts: 2
  Max tokens: 200 | Temperature: 0.9

Building multi-adapter model...
  Creating PeftModel from epoch 3 (adapter: 'epoch_3')


c:\Users\baruc\Desktop\Preference_Trees\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


  Loading adapter for epoch 6 (adapter: 'epoch_6')
  Loading adapter for epoch 9 (adapter: 'epoch_9')
  Loading adapter for epoch 12 (adapter: 'epoch_12')
  Loading adapter for epoch 15 (adapter: 'epoch_15')
  ✓ Multi-adapter model ready with 5 adapter(s)
    Available adapters: ['epoch_3', 'epoch_6', 'epoch_9', 'epoch_12', 'epoch_15']
✓ Model ready (adapters enabled: True)


[Prompt 1]
Input(truncated): ...the rest of my life. <|im_end|>
<|im_start|> user
That sounds a bit extreme, honestly. Seeing a doctor after every smoking session? It feels more like a punishment than a solution. I mean, if I have to keep going to the doctor just to keep smoking, what’s the point? It sounds like a lot of hassle for something I’m already worried about. I guess I need to think about whether I actually want to make that effort for my health or if I’m just going to keep doing what I’m doing. <|im_end|>
<|im_start|> assistant

----------------------------------------------------------------------------

KeyboardInterrupt: 

In [ ]:
# clean up cuda memory after sweep
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# print model adaptors
if hasattr(base_policy, "peft_config"):
    print("Available adapters in base_policy:")
    for name, config in base_policy.peft_config.items():
        print(f"  - {name}: {config}")
else:
    print("No adapters found in base_policy.") 

Available adapters in base_policy:
  - epoch_3: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path='meta-llama/Llama-3.2-1B', revision=None, inference_mode=True, r=16, target_modules={'gate_proj', 'k_proj', 'q_proj', 'o_proj', 'up_proj', 'down_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)
  - epoch_6: LoraConf